# RLAIF with PPO — Aligning an LLM to AI-Generated Preferences

## What is RLAIF (and how it relates to RLHF)?

**RLAIF (Reinforcement Learning from AI Feedback)** uses the *exact same* optimization machinery as **RLHF** — a reward model plus PPO with a KL penalty — but the preference labels that train the reward model are produced by an **AI judge (a strong LLM)** rather than by human annotators. Everything downstream of the labels is identical; only the **source of the feedback** changes.

> 🔑 **Why this notebook is RLAIF, not RLHF:** its reward model is trained on **`trl-lib/ultrafeedback_binarized`**, whose `chosen`/`rejected` labels were generated by **GPT-4 scoring model outputs** (from *UltraFeedback: Boosting Language Models with Scaled **AI** Feedback*) — no human raters were involved. Swap that dataset for a human-labelled one (e.g. `Anthropic/hh-rlhf`) and the identical code becomes genuine RLHF — see the companion **`RLHF_with_PPO.ipynb`**.

## Technical Terminology & Mechanics

**Proximal Policy Optimization (PPO)** optimizes an LLM (the **Actor / Policy**) to maximize a scalar reward emitted by a **separately trained Reward Model**. It does so with a **clipped surrogate objective** (the *trust region*) and **Generalized Advantage Estimation (GAE)** for low-variance advantage estimates from a learned value function (the **Critic**), all while a **KL-divergence penalty** keeps the policy from drifting away from its supervised (SFT) starting point.

### Crisp Definition

PPO is an **on-policy reinforcement learning algorithm** that fine-tunes an SFT model by:

1. Sampling completions and **scoring them with a pre-trained Reward Model**
2. Estimating advantages with a **learned value function (Critic)** + GAE
3. Updating the policy via a **clipped surrogate objective**, regularized by a **KL penalty** so it does not catastrophically forget its SFT language distribution

### The KL-Penalty Objective Formulation

To stop the policy from **"gaming"** the reward model (e.g., emitting repetitive gibberish that happens to score high), the reward $R_{\phi}$ is penalized by the KL divergence from the frozen reference policy $\pi_{\text{ref}}$, scaled by $\beta$:

$$R_{\text{total}}(x, y) = R_{\phi}(x, y) - \beta \log \left( \frac{\pi_{\theta}(y|x)}{\pi_{\text{ref}}(y|x)} \right)$$

---

## The Feedback Source: AI-Generated Preferences

Unlike Supervised Fine-Tuning (which requires full completions), the PPO loop itself only needs **unlabelled prompts**. All preference information is baked into the Reward Model *offline*, before PPO starts — and in RLAIF those preferences come from an **AI judge**.

1. **`trl-lib/ultrafeedback_binarized`** — an **AI-feedback** preference dataset structured as `(prompt, chosen, rejected)`, where the `chosen`/`rejected` verdicts were produced by **GPT-4**. The **Reward Model** learns the *AI preference delta* between the chosen and rejected response. *(This notebook trains its Reward Model here.)*

2. **`tatsu-lab/alpaca`** — once the Reward Model is trained, PPO only needs a diverse set of **input contexts**. We use Alpaca's **instruction** prompts (write / explain / brainstorm / classify …), which match the UltraFeedback reward's *general-helpfulness* competence far better than narrow Reddit summarization. It lets the policy:
    - Explore novel completions ($y$) for a given state ($x$)
    - Receive real-time scoring from the frozen Reward Model

> **Reward Model vs. Critic — do not conflate them.** The **Reward Model** is trained *offline* on the AI-generated preferences and stays **frozen** during PPO. The **Critic** (value head) is trained *online during PPO* to predict expected return and is only used to compute advantages. Two different models, two different jobs.

---

# Architectural Context Block

## The "Why"

SFT suffers from **exposure bias** and merely clones the average behavior of a static dataset. RLAIF (like RLHF) via PPO shifts the paradigm from **imitation** to **exploration** — the model discovers sequences that maximize the reward, while a clipping threshold ($\epsilon$) and KL penalty keep updates from destroying the SFT baseline.

---

## VRAM & Compute Impact

PPO is **memory-hungry**. A naive implementation keeps **four models** resident at once:

| Model | Role |
|---|---|
| **Actor** | The policy being actively updated |
| **Critic / Value Head** | Estimates state-values; trained *during* PPO |
| **Reference Model** | Frozen SFT policy used for the KL penalty |
| **Reward Model** | Frozen model that scores completions |

On a **16 GB T4**, this is only feasible with parameter-efficient tricks (QLoRA), plus the adapter-disabling trick below that removes the 4th model.

---

## Architectural Trade-offs

### ✅ Pros
- **Scalable feedback** — no human labelling bottleneck; an AI judge labels preferences cheaply
- Actively **curbs low-quality output**
- Highly **customizable** via reward-function / judge-prompt engineering

### ❌ Cons
- **Inherits the judge's biases** — the policy can only be as well-aligned as the AI labeller
- **Hyperparameter sensitivity:** very sensitive to learning rate, $\beta$ (KL), and batch size
- **Reward hacking:** catastrophic if the Reward Model is weak or *untrained*
- **Engineering overhead:** far more complex than offline methods like DPO


# Production-Grade Implementation (Colab T4, 16 GB)

To fit this on a **Google Colab T4 GPU (16 GB VRAM)** we use **4-bit Quantization (QLoRA)** for every model, and lean on TRL's ability to reuse the policy as its own reference.

> ⚙️ **Architectural lifesaver — reference model via adapter disabling:** Instead of loading a **separate reference model** (another ~1 GB), passing `ref_model=None` to a PEFT policy makes TRL compute the KL penalty by **temporarily disabling the LoRA adapters** — i.e., the frozen base weights *are* the reference. This removes the 4th model from memory and makes T4-scale PPO feasible.

**The pipeline below runs in 5 stages:**

| Stage | What | Why |
|---|---|---|
| 1 | Use `Qwen2.5-0.5B-**Instruct**` as the policy | SFT is already done — gives coherent behavior to reinforce |
| 2 | Train a **Reward Model** (`RewardTrainer`) on **AI-labelled** pairs | Turns GPT-4-generated preferences into a scalar signal PPO can chase |
| 3 | Assemble Actor + Critic + Reward + (implicit) Reference | The four PPO components |
| 4 | Prepare **prompt-only** PPO data (`tatsu-lab/alpaca` instructions) | PPO explores completions for these contexts |
| 5 | **PPO** train, then evaluate | Align the policy to the reward |


## Environment Setup

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
%pip install torchao==0.16.0 transformers trl peft accelerate bitsandbytes datasets

In [ ]:
import torch
import os
import gc
from transformers import AutoTokenizer, DataCollatorWithPadding, BitsAndBytesConfig, set_seed, AutoModelForCausalLM, AutoModelForSequenceClassification, GenerationConfig
from trl import RewardConfig, RewardTrainer
from trl.experimental.ppo import PPOConfig, PPOTrainer
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset

set_seed(42)
# Prevents CUDA fragmentation OOMs on Colab T4
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## Setup — Quantization, LoRA & Tokenizer

We start from the **Instruct** checkpoint (this *is* Stage 1 / SFT — Qwen already fine-tuned it to follow instructions), define one 4-bit config shared by every model, and two separate LoRA configs: one for the **policy** (a Causal LM) and one for the **reward model** (a Sequence-Classification model, which additionally needs its `score` head trained).

In [ ]:
# IMPORTANT: start from the *Instruct* checkpoint, not the raw base model.
# RLAIF (like RLHF) assumes the policy has already been through SFT ("learned how to talk").
# Qwen2.5-0.5B-Instruct IS that SFT stage — it can already follow a "summarize this"
# instruction, giving PPO coherent behavior to reinforce AND a coherent KL reference.
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 1. 4-bit Quantization (shared by every model to fit the T4's 16 GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 2a. LoRA for the POLICY (a CausalLM) — no score head here.
policy_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
)

# 2b. LoRA for the REWARD MODEL (a SequenceClassification model).
#     modules_to_save=["score"] is ESSENTIAL: the scalar reward head starts random,
#     so it must be trainable (not frozen) or the reward signal stays meaningless.
reward_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    bias="none",
    task_type="SEQ_CLS",
    modules_to_save=["score"],
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## Stage 2 — Reward Model Training (on AI Feedback)

The Reward Model is the heart of RLAIF: it converts the **AI-generated** `(chosen, rejected)` preferences — GPT-4's judgments from UltraFeedback — into a scalar the policy can optimize. We train it with TRL's `RewardTrainer` under the **Bradley–Terry** objective, which maximizes the score gap between chosen and rejected responses.

Watch the logged **`accuracy`** climb above `0.5` — that is your proof the reward head has learned a *real* signal (not noise). A random reward head is fatal: PPO would faithfully maximize noise, producing incoherent, off-task text.

> 💡 **Why the *value* head can stay random but the *reward* head cannot:** the value/critic head (Stage 3) is **learned during PPO** from observed returns, so random init is expected. The reward head is **frozen** during PPO — it must arrive already trained.


In [ ]:
# Small slice of an AI-feedback preference dataset (UltraFeedback = GPT-4 as judge).
# Keeps this ~10 min on a T4.
rm_dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")
rm_dataset = rm_dataset.shuffle(seed=42).select(range(800))

reward_config = RewardConfig(
    output_dir="./reward_model_adapter",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=1e-4,  # higher LR is fine — only LoRA + head are trained
    num_train_epochs=1,
    max_length=512,  # filters long pairs; keeps VRAM in budget
    logging_steps=10,
    bf16=True, fp16=False,  # KEY: T4 runs bf16 (emulated) and bf16 uses NO GradScaler,
                            # so the "unscale not implemented for BFloat16" error is gone.
                            # (fp16=True was the bug: its fp16 scaler can't unscale the
                            # bf16 grads that Colab's bf16 autocast produces.)
    center_rewards_coefficient=1e-2,  # keeps rewards ~zero-centered, curbs reward hacking
    report_to="none",
)

reward_trainer = RewardTrainer(
    model=base_model_id,  # loaded as AutoModelForSequenceClassification, num_labels=1
    args=reward_config,
    train_dataset=rm_dataset,
    processing_class=tokenizer,
    peft_config=reward_lora_config,  # trains LoRA + the "score" head
    quantization_config=bnb_config,  # QLoRA: 4-bit base
)
reward_trainer.train()

# 'accuracy' in the logs should climb well above 0.5 — a real signal, not noise.
reward_trainer.save_model("./reward_model_adapter")

# Free the trainer/optimizer state before PPO loads its 3 models.
del reward_trainer
gc.collect()
torch.cuda.empty_cache()

## Stage 3 — Assemble the PPO Models

Three models are instantiated (the fourth — the **reference** — is the policy with adapters disabled, via `ref_model=None`):

- **Actor / Policy** — Instruct Causal LM + trainable LoRA.
- **Critic / Value model** — a fresh Sequence-Classification head. A *random* value head is **correct** here; it is learned during PPO.
- **Reward model** — the model we just trained in Stage 2, loaded frozen.

In [ ]:
# A) ACTOR (Policy): Instruct CausalLM + QLoRA adapters.
#    With ref_model=None, TRL uses this same model with adapters DISABLED as the
#    frozen KL reference — i.e. the coherent Instruct model.
actor_model = AutoModelForCausalLM.from_pretrained(
    base_model_id, quantization_config=bnb_config, device_map="auto"
)
actor_model = get_peft_model(actor_model, policy_lora_config)
actor_model.config.use_cache = False
actor_model.config.pad_token_id = tokenizer.pad_token_id
actor_model.generation_config = GenerationConfig(
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id
)

# B) CRITIC (Value Model): fresh SequenceClassification head.
#    A RANDOM value head is CORRECT here — the value function is learned during PPO.
value_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_id, num_labels=1, quantization_config=bnb_config, device_map="auto"
)
value_model.config.pad_token_id = tokenizer.pad_token_id  # seq-cls head needs this to
value_model.config.use_cache = False  # find the last non-pad token

# C) REWARD MODEL: the model we trained in Stage 2 (base + trained LoRA + score head).
#    Frozen during PPO. This is the real, meaningful reward signal.
reward_base = AutoModelForSequenceClassification.from_pretrained(
    base_model_id, num_labels=1, quantization_config=bnb_config, device_map="auto"
)
reward_model = PeftModel.from_pretrained(reward_base, "./reward_model_adapter")
reward_model.eval()
reward_model.config.pad_token_id = tokenizer.pad_token_id
reward_model.config.use_cache = False

## Stage 4 — Dataset Preparation (PPO prompts)

PPO consumes **prompt-only** data. We use **`tatsu-lab/alpaca`** instruction prompts (write / explain / brainstorm / classify …) instead of Reddit summarization.

> ⚠️ **Why the task changed:** the reward model was trained on **UltraFeedback**, which scores *general helpfulness*, not *summarization faithfulness*. Optimizing summarization against it just makes text "sound helpful" (fluent, generic, mildly hallucinated). Matching the PPO task to the reward model's competence — open-ended instruction-following — is what makes the reward signal meaningful.

Each instruction is wrapped in the **Instruct chat template** (`add_generation_prompt=True`) so the policy emits a proper assistant turn.

In [ ]:
# Prompt-only PPO data: diverse instructions from Alpaca (matches the UltraFeedback
# reward model's "general helpfulness" competence far better than TL;DR summarization).
raw_dataset = load_dataset("tatsu-lab/alpaca", split="train").shuffle(seed=42).select(range(1500))

def build_prompt(example):
    instruction = (example["instruction"] or "").strip()
    context = (example.get("input") or "").strip()
    user_content = instruction if not context else f"{instruction}\n\n{context}"
    if not user_content:
        return {"input_ids": [], "attention_mask": [], "keep": False}
    messages = [{"role": "user", "content": user_content}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, truncation=True, max_length=384)
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "keep": True}

tokenized_dataset = raw_dataset.map(build_prompt, remove_columns=raw_dataset.column_names)
tokenized_dataset = tokenized_dataset.filter(lambda ex: ex["keep"])
tokenized_dataset = tokenized_dataset.select_columns(["input_ids", "attention_mask"])
tokenized_dataset = tokenized_dataset.select(range(min(300, len(tokenized_dataset))))
print(f"PPO prompts: {len(tokenized_dataset)}")

# Dynamic padding collator (saves VRAM vs. padding to max_length)
collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")

## Stage 5 — PPO Training

In [ ]:
# PPO Config
ppo_config = PPOConfig(
    # --- Standard TrainingArguments Base Parameters ---
    output_dir="./ppo_output",
    run_name="ppo-rlaif-t4",
    learning_rate=1e-5,      # low enough to be stable, high enough to actually move the policy
    logging_steps=1,

    # --- Hardened VRAM Constraints (Base Class) ---
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=True, fp16=False,   # bf16 => no fp16 GradScaler => no bf16-unscale crash

    # --- TRL Rollout & Generation Parameters ---
    # Crucial for T4: keeps generation batch size small to avoid KV-Cache OOMs
    local_rollout_forward_batch_size=2,
    response_length=64,
    temperature=0.7,
    stop_token="eos",
    missing_eos_penalty=1.0,  # nudges completions to actually terminate

    # --- TRL PPO Mathematical Step Parameters ---
    num_mini_batches=1,
    num_ppo_epochs=2,  # reuse each rollout batch twice
    whiten_rewards=True,  # whitens a REAL reward now, not random noise
    num_train_epochs=1,

    # --- Alignment & Trust Region Constraints ---
    kl_coef=0.1,  # lets the policy improve while the (coherent) Instruct
                  # reference keeps it from drifting into gibberish
    kl_estimator="k3",
    cliprange=0.2,
    cliprange_value=0.2,
    vf_coef=0.1,
    gamma=1.0,
    lam=0.95,
)

# TRAINER INIT
ppo_trainer = PPOTrainer(
    args=ppo_config,
    model=actor_model,
    ref_model=None,  # None => KL reference = Instruct base (adapters disabled)
    reward_model=reward_model,
    value_model=value_model,
    processing_class=tokenizer,
    train_dataset=tokenized_dataset,
    data_collator=collator,
)

In [ ]:
ppo_trainer.train()

# Save the aligned adapter
ppo_trainer.save_model("./rlaif_with_ppo_adapter")

## Export — Download the Fine-Tuned Adapter (Optional)

In [ ]:
import shutil
import os

# Configuration
folder_to_zip = './rlaif_with_ppo_adapter'
output_filename = 'rlaif_with_ppo_adapter.zip'

# Create the zip archive
shutil.make_archive(output_filename.replace('.zip', ''), 'zip', folder_to_zip)

if os.path.exists(output_filename):
    file_size = os.path.getsize(output_filename)
    print(f"File: {output_filename}")
    print(f"Size in MB: {file_size / (1024 * 1024):.2f} MB")
    print(f"Size in GB: {file_size / (1024 * 1024 * 1024):.2f} GB")
else:
    print(f"File {output_filename} not found. Please ensure the zipping cell above has been executed.")

### Download the file to your machine

In [ ]:
from google.colab import files
files.download(output_filename)

### Save the Adapter to Google Drive

In [ ]:
from google.colab import drive

# 1. Mount Google Drive
# This will open a pop-up for authorization
drive.mount('/content/drive')

# Configuration
filename = 'reward_model_adapter.zip'
destination_folder = '/content/drive/MyDrive/colab_models'

if os.path.exists(folder_to_zip):
        # Ensure destination exists in Drive
        os.makedirs(destination_folder, exist_ok=True)
        destination_path = os.path.join(destination_folder, output_filename)

        # Copy the file
        print(f"Copying to Drive: {destination_path}...")
        shutil.copy(output_filename, destination_path)
        print("Done! The model is now backed up to your Google Drive.")
else:
    print(f"Error: Source folder {folder_to_zip} not found. Did the training finish?")

# Model Usage — Evaluate the Aligned Policy

We reload the **Instruct** base + the PPO-tuned LoRA adapter, and generate with the **same chat-template formatting** used during training. (Matching the training-time prompt format is not optional — feeding raw prompts to an Instruct model produces rambling output regardless of how well PPO trained.)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# 1. Setup paths and configurations
#    MUST match the policy base used in training: the Instruct checkpoint.
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_path = "./rlaif_with_ppo_adapter"

# T4 VRAM Optimization for loading
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 2. Load Tokenizer & Quantized Base Model
print("Loading base tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto"
)

# 3. Apply the PPO-tuned LoRA Adapter
print("Applying PPO-tuned LoRA adapters...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()  # Set model to evaluation mode

In [ ]:
# INFERENCE UTILITY — same chat template the PPO loop used (a single user instruction).
def generate_response(instruction, max_new_tokens=200, temperature=0.7):
    messages = [{"role": "user", "content": instruction}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    prompt_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)

In [ ]:
# RUN TEST PROMPTS (instruction-following — matches the reward model's competence)
test_prompts = [
    "Give me three practical tips for staying focused while working from home.",
    "Explain what reinforcement learning is to a 12-year-old in a few sentences.",
]

print("\n--- Generating Aligned Responses ---")
for i, prompt in enumerate(test_prompts):
    print(f"\n[Instruction {i+1}]: {prompt}")
    print(f"[RLAIF Aligned Response]: {generate_response(prompt).strip()}")
    print("-" * 50)